# **Training**

In [ ]:
import os
import shutil
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
import timm
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
from tqdm.auto import tqdm
from sklearn.metrics import f1_score as sk_f1

torch.manual_seed(42)
np.random.seed(42)

# =========================
# CONFIGURATION
# =========================
SIZE_MODE    = "384"
POOLING_MODE = "gated_attention"

TRAIN_CSV = "/content/drive/MyDrive/FYP/natural foci ground truths/data/culture_only_v02/train_dataset.csv"
VAL_CSV   = "/content/drive/MyDrive/FYP/natural foci ground truths/data/culture_only_v02/val_dataset.csv"
TEST_CSV  = "/content/drive/MyDrive/FYP/natural foci ground truths/data/culture_only_v02/test_dataset.csv"

CFG = {
    "batch_size": 256,
    "epochs":     15,

    "lr_head": 1e-3,
    "wd":      1e-4,

    "train_csv": TRAIN_CSV,
    "val_csv":   VAL_CSV,

    "backbone_ckpt": "/content/drive/MyDrive/FYP/natural foci ground truths/backbone models/CELL_ONLY_foci_dino_backbone_DINOv03_v01.pth",
    "save_path":  "/content/drive/MyDrive/FYP/natural foci ground truths/models/final_models/finetune_gated_attention_cell_only_V001_frozen/ablation_frozen_DINO_gated_LN_nopw/baseline_model.pth",
    "output_dir": "/content/drive/MyDrive/FYP/natural foci ground truths/models/final_models/finetune_gated_attention_cell_only_V001_frozen/ablation_frozen_DINO_gated_LN_nopw/plots",
    "device": torch.device("cuda" if torch.cuda.is_available() else "cpu")
}

if SIZE_MODE == "224":
    CFG.update({"image_size": 224, "backbone_name": "vit_tiny_patch16_224"})
elif SIZE_MODE == "384":
    CFG.update({"image_size": 384, "backbone_name": "vit_tiny_patch16_384"})

DEVICE = CFG["device"]
print(f"--- ABLATION D: No pos_weight (Plain BCE Loss) ---")
print(f"Pooling Strategy : {POOLING_MODE.upper()}")
print(f"Backbone Init    : DINO pretrained")
print(f"Backbone         : Fully frozen for all {CFG['epochs']} epochs")
print(f"Loss             : BCEWithLogitsLoss — NO pos_weight")

# =========================
# DATA AUGMENTATIONS & DATASET
# =========================

NORM_MEAN = [0.2251, 0.2251, 0.2251]
NORM_STD  = [0.2375, 0.2375, 0.2375]

def get_transforms(img_size, is_train=True):
    if is_train:
        return transforms.Compose([
            transforms.Resize((img_size, img_size), interpolation=transforms.InterpolationMode.BICUBIC),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.RandomVerticalFlip(p=0.5),
            transforms.RandomRotation(degrees=15),
            transforms.RandomApply([transforms.ColorJitter(brightness=0.1, contrast=0.1)], p=0.3),
            transforms.ToTensor(),
            transforms.Normalize(mean=NORM_MEAN, std=NORM_STD)
        ])
    else:
        return transforms.Compose([
            transforms.Resize((img_size, img_size), interpolation=transforms.InterpolationMode.BICUBIC),
            transforms.ToTensor(),
            transforms.Normalize(mean=NORM_MEAN, std=NORM_STD)
        ])

class FociDataset(Dataset):
    def __init__(self, csv_path, img_size=224, is_train=True):
        self.df        = pd.read_csv(csv_path)
        self.transform = get_transforms(img_size, is_train)
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        try:
            img_p = str(row['image_path'])
            img   = Image.open(img_p).convert("RGB")
            x     = self.transform(img)
            y     = float(row["label"])
            return x, torch.tensor(y).float()
        except Exception: return None

def safe_collate(batch):
    batch = [b for b in batch if b is not None]
    if not batch: return None
    xs, ys = zip(*batch)
    return torch.stack(xs), torch.stack(ys)

In [ ]:
# =========================
# MODEL
# =========================
class FociViT_Baseline(nn.Module):
    def __init__(self, backbone_name, backbone_ckpt, img_size=224, pooling_mode="gated_attention"):
        super().__init__()
        self.pooling_mode = pooling_mode

        self.backbone = timm.create_model(
            backbone_name, pretrained=False,
            num_classes=0, global_pool="", img_size=img_size
        )

        if os.path.exists(backbone_ckpt):
            sd = torch.load(backbone_ckpt, map_location="cpu")
            self.backbone.load_state_dict(
                {k.replace("backbone.", "").replace("vit.", ""): v for k, v in sd.items()},
                strict=False
            )
            print("Loaded DINO Backbone Weights.")

        for param in self.backbone.parameters():
            param.requires_grad = False
        print("Backbone fully frozen for all epochs.")

        with torch.no_grad():
            dim = self.backbone.forward_features(
                torch.zeros(1, 3, img_size, img_size)
            ).shape[-1]

        if self.pooling_mode == "gated_attention":
            hidden_dim       = 128
            self.head_norm   = nn.LayerNorm(dim)
            self.attention_V = nn.Sequential(nn.Linear(dim, hidden_dim), nn.Tanh())
            self.attention_U = nn.Sequential(nn.Linear(dim, hidden_dim), nn.Sigmoid())
            self.attention_w = nn.Linear(hidden_dim, 1)

        self.classifier = nn.Sequential(
            nn.Linear(dim, 128), nn.ReLU(), nn.Dropout(0.3), nn.Linear(128, 1)
        )

    def forward(self, x):
        feats = self.backbone.forward_features(x)
        if isinstance(feats, dict): feats = feats["x"]
        patch_tokens = feats[:, 1:, :]

        if self.pooling_mode == "gated_attention":
            normed_patches  = self.head_norm(patch_tokens)
            A_V             = self.attention_V(normed_patches)
            A_U             = self.attention_U(normed_patches)
            A_raw           = self.attention_w(A_V * A_U).squeeze(-1)
            A               = torch.softmax(A_raw, dim=1)
            pooled_features = torch.sum(patch_tokens * A.unsqueeze(-1), dim=1)
        else:
            pooled_features = feats[:, 0, :]

        logits = self.classifier(pooled_features).squeeze(-1)
        return logits


In [ ]:
# =========================
# TRAINING LOOP
# =========================
def main():
    os.makedirs(os.path.dirname(CFG["save_path"]), exist_ok=True)
    os.makedirs(CFG["output_dir"], exist_ok=True)

    train_loader = DataLoader(
        FociDataset(CFG["train_csv"], CFG["image_size"], is_train=True),
        batch_size=CFG["batch_size"], shuffle=True,
        collate_fn=safe_collate, num_workers=0
    )
    val_loader = DataLoader(
        FociDataset(CFG["val_csv"], CFG["image_size"], is_train=False),
        batch_size=CFG["batch_size"], shuffle=False,
        collate_fn=safe_collate, num_workers=0
    )

    model = FociViT_Baseline(
        CFG["backbone_name"], CFG["backbone_ckpt"],
        CFG["image_size"], POOLING_MODE
    ).to(DEVICE)

    head_params = list(model.classifier.parameters())
    if POOLING_MODE == "gated_attention":
        head_params += (
            list(model.head_norm.parameters())   +
            list(model.attention_V.parameters()) +
            list(model.attention_U.parameters()) +
            list(model.attention_w.parameters())
        )

    optimizer = torch.optim.AdamW(
        [{"params": head_params, "lr": CFG["lr_head"]}],
        weight_decay=CFG["wd"]
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=CFG["epochs"], eta_min=1e-6
    )

    criterion = nn.BCEWithLogitsLoss()

    label_counts = pd.read_csv(TRAIN_CSV)["label"].value_counts()
    n0 = label_counts[0]
    n1 = label_counts[1]
    print(f"Label counts — 0: {n0}, 1: {n1}")

    history = {
        "train_loss": [], "val_loss": [],
        "train_acc":  [], "val_acc":  [],
        "val_f1":     [], "lr_head":  []
    }
    best_f1 = 0.0

    print("\nStarting Training...")
    for epoch in range(CFG["epochs"]):

        model.train()
        t_loss, t_correct, t_total = 0.0, 0, 0

        for batch in tqdm(train_loader, desc=f"Epoch {epoch+1} [Train]", leave=False):
            if batch is None: continue
            x, y = [b.to(DEVICE) for b in batch]
            optimizer.zero_grad()
            logits = model(x)
            loss   = criterion(logits, y)
            loss.backward()
            optimizer.step()

            B          = x.size(0)
            t_loss    += loss.item() * B
            t_correct += ((logits >= 0.0).float() == y).sum().item()
            t_total   += B

        model.eval()
        v_loss, v_correct, v_total = 0.0, 0, 0
        v_preds_all, v_labels_all  = [], []

        with torch.no_grad():
            for batch in val_loader:
                if batch is None: continue
                x, y   = [b.to(DEVICE) for b in batch]
                logits = model(x)
                loss   = criterion(logits, y)

                B          = x.size(0)
                v_loss    += loss.item() * B
                v_correct += ((logits >= 0.0).float() == y).sum().item()
                v_total   += B

                v_preds_all.extend((logits >= 0.0).float().cpu().numpy())
                v_labels_all.extend(y.cpu().numpy())

        scheduler.step()

        lr_now         = optimizer.param_groups[0]["lr"]
        train_loss_avg = t_loss    / t_total
        val_loss_avg   = v_loss    / max(v_total, 1)
        train_acc_avg  = t_correct / t_total
        val_acc_avg    = v_correct / max(v_total, 1)
        val_f1         = sk_f1(v_labels_all, v_preds_all, zero_division=0)

        history["train_loss"].append(train_loss_avg)
        history["val_loss"].append(val_loss_avg)
        history["train_acc"].append(train_acc_avg)
        history["val_acc"].append(val_acc_avg)
        history["val_f1"].append(val_f1)
        history["lr_head"].append(lr_now)

        print(
            f"Epoch {epoch+1:02d}: "
            f"Train Loss={train_loss_avg:.4f}, Acc={train_acc_avg:.4f} | "
            f"Val Loss={val_loss_avg:.4f}, Acc={val_acc_avg:.4f}, F1={val_f1:.4f} | "
            f"LR: {lr_now:.2e}"
        )

        if val_f1 >= best_f1:
            best_f1 = val_f1
            torch.save(model.state_dict(), CFG["save_path"])
            print(f"Saved Best Model (Val F1: {best_f1:.4f})")

        torch.save(model.state_dict(), f"/content/ckpt_epoch_{epoch+1}.pth")

    print("\nCopying per-epoch checkpoints to Drive...")
    for ep in range(1, CFG["epochs"] + 1):
        local_ckpt = f"/content/ckpt_epoch_{ep}.pth"
        if os.path.exists(local_ckpt):
            shutil.copy2(local_ckpt, CFG["save_path"].replace(".pth", f"_epoch_{ep}.pth"))
            os.remove(local_ckpt)
    print("All checkpoints copied to Drive.")

    # ── Plot ──────────────────────────────────────────────────────────────────
    fig, ax = plt.subplots(1, 3, figsize=(18, 5))
    epochs_range = range(1, CFG["epochs"] + 1)

    ax[0].plot(epochs_range, history["train_loss"], label="Train Loss", marker="o")
    ax[0].plot(epochs_range, history["val_loss"],   label="Val Loss",   marker="o")
    ax[0].set_title("BCE Loss — Ablation D (No pos_weight)")
    ax[0].legend(); ax[0].grid(True, alpha=0.3)

    ax[1].plot(epochs_range, history["train_acc"], label="Train Acc", marker="o")
    ax[1].plot(epochs_range, history["val_acc"],   label="Val Acc",   marker="o")
    ax[1].plot(epochs_range, history["val_f1"],    label="Val F1",    marker="o", color="green")
    ax[1].set_title("Accuracy & F1 — Ablation D (No pos_weight)")
    ax[1].legend(); ax[1].grid(True, alpha=0.3)

    ax[2].plot(epochs_range, history["lr_head"], label="Head LR", marker="o", color="blue")
    ax[2].set_title("LR Schedule (Head Only)")
    ax[2].set_yscale("log"); ax[2].legend(); ax[2].grid(True, alpha=0.3)

    plt.tight_layout()
    plot_path = f"{CFG['output_dir']}/training_metrics_ablation_D_nopw.png"
    plt.savefig(plot_path)
    print(f"\nTraining Complete! Best Val F1: {best_f1:.4f}")
    print(f"Plot saved to {plot_path}")


if __name__ == "__main__":
    main()

# **Evaluation**

In [ ]:
import os
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
import timm
import matplotlib.pyplot as plt
import seaborn as sns
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
from tqdm.auto import tqdm
from sklearn.decomposition import PCA
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, roc_curve, auc
)

# =========================
# CONFIGURATION
# =========================
SIZE_MODE    = "384"
POOLING_MODE = "gated_attention"

SAVE_VISUALIZATIONS = True
NUM_VIS_TO_SAVE     = 30

TEST_CSV   = "/content/drive/MyDrive/FYP/natural foci ground truths/data/culture_only_v02/test_dataset.csv"

CFG = {
    "batch_size":   32,
    "test_csv":     TEST_CSV,
    "model_ckpt":   f"/content/drive/MyDrive/FYP/natural foci ground truths/models/final_models/finetune_gated_attention_cell_only_V001_frozen/No pos weight/ablation_frozen_ImageNet_gated_LN_pw/baseline_model.pth",
    "output_dir":   f"/content/drive/MyDrive/FYP/natural foci ground truths/models/final_models/finetune_gated_attention_cell_only_V001_frozen/No pos weight/ablation_frozen_ImageNet_gated_LN_pw/eval_results/test_v01",
    "image_size":   int(SIZE_MODE),
    "patch_size":   16,
    "backbone_name": f"vit_tiny_patch16_{SIZE_MODE}",
    "device":       torch.device("cuda" if torch.cuda.is_available() else "cpu"),
}

GRID_SIZE = CFG["image_size"] // CFG["patch_size"]   # 24
DEVICE    = CFG["device"]

os.makedirs(CFG["output_dir"], exist_ok=True)
print(f"--- EVALUATION: {POOLING_MODE.upper()} (V02) ---")
print(f"Checkpoint : {CFG['model_ckpt']}")
print(f"Test CSV   : {CFG['test_csv']}")

# =========================
# DATASET
# =========================

NORM_MEAN = [0.2251, 0.2251, 0.2251]
NORM_STD  = [0.2375, 0.2375, 0.2375]

eval_transform = transforms.Compose([
    transforms.Resize((CFG["image_size"], CFG["image_size"]),
                      interpolation=transforms.InterpolationMode.BICUBIC),
    transforms.ToTensor(),
    transforms.Normalize(mean=NORM_MEAN, std=NORM_STD)
])

class FociEvalDataset(Dataset):
    def __init__(self, csv_path):
        self.df = pd.read_csv(csv_path)
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        try:
            img_p = str(row["image_path"])
            img   = Image.open(img_p).convert("RGB")
            x     = eval_transform(img)
            y     = float(row["label"])
            return x, torch.tensor(y).float(), img_p
        except Exception: return None

def safe_collate_eval(batch):
    batch = [b for b in batch if b is not None]
    if not batch: return None
    xs, ys, paths = zip(*batch)
    return torch.stack(xs), torch.stack(ys), list(paths)

# =========================
# EVAL MODEL
# =========================

class FociViT_Eval(nn.Module):
    def __init__(self, backbone_name, img_size=384, pooling_mode="gated_attention"):
        super().__init__()
        self.pooling_mode = pooling_mode

        self.backbone = timm.create_model(
            backbone_name, pretrained=False,
            num_classes=0, global_pool="", img_size=img_size
        )

        with torch.no_grad():
            dim = self.backbone.forward_features(
                torch.zeros(1, 3, img_size, img_size)
            ).shape[-1]

        if self.pooling_mode == "gated_attention":
            hidden_dim = 128
            self.head_norm   = nn.LayerNorm(dim)
            self.attention_V = nn.Sequential(nn.Linear(dim, hidden_dim), nn.Tanh())
            self.attention_U = nn.Sequential(nn.Linear(dim, hidden_dim), nn.Sigmoid())
            self.attention_w = nn.Linear(hidden_dim, 1)

        self.classifier = nn.Sequential(
            nn.Linear(dim, 128), nn.ReLU(), nn.Dropout(0.3), nn.Linear(128, 1)
        )

    def forward(self, x):
        feats = self.backbone.forward_features(x)
        if isinstance(feats, dict): feats = feats["x"]

        patch_tokens = feats[:, 1:, :]   # (B, 576, dim)
        spatial_map  = None

        if self.pooling_mode == "gated_attention":
            normed_patches = self.head_norm(patch_tokens)

            A_V   = self.attention_V(normed_patches)
            A_U   = self.attention_U(normed_patches)
            A_raw = self.attention_w(A_V * A_U).squeeze(-1)
            A     = torch.softmax(A_raw, dim=1)   # (B, 576)

            pooled_features = torch.sum(patch_tokens * A.unsqueeze(-1), dim=1)
            spatial_map     = A
        else:
            pooled_features = feats[:, 0, :]

        logits = self.classifier(pooled_features).squeeze(-1)
        probs  = torch.sigmoid(logits)

        return probs, spatial_map, patch_tokens

# =========================
# DASHBOARD VISUALIZATION
# =========================
def norm_01(arr):
    return (arr - arr.min()) / (arr.max() - arr.min() + 1e-8)

def save_dashboard(img_path, patch_tokens, attn_map, prob, label, idx, out_dir):
    pil_img     = Image.open(img_path).convert("RGB")
    img_resized = pil_img.resize((CFG["image_size"], CFG["image_size"]))
    tokens_np   = patch_tokens.cpu().numpy()   # (576, dim)
    gt_label    = "Damaged"  if int(label) == 1 else "Healthy"
    pred_label  = "Damaged"  if prob >= 0.5    else "Healthy"
    correct     = "✓" if pred_label == gt_label else "✗"

    # ── Norm map ─────────────────────────────────────────────────────────────
    norms    = np.linalg.norm(tokens_np, axis=-1)
    norm_map = norm_01(norms.reshape(GRID_SIZE, GRID_SIZE))

    # ── PCA RGB ───────────────────────────────────────────────────────────────
    pca     = PCA(n_components=3)
    pca_out = pca.fit_transform(tokens_np)
    for c in range(3): pca_out[:, c] = norm_01(pca_out[:, c])
    pca_rgb = pca_out.reshape(GRID_SIZE, GRID_SIZE, 3)

    # ── Attention map ─────────────────────────────────────────────────────────
    attn_2d  = norm_01(attn_map.cpu().numpy().reshape(GRID_SIZE, GRID_SIZE))
    attn_img = Image.fromarray((attn_2d * 255).astype(np.uint8)).resize(
        (CFG["image_size"], CFG["image_size"]), Image.BILINEAR  # match img_resized exactly
    )

    fig, axes = plt.subplots(2, 3, figsize=(15, 10))

    # Row 0 — backbone view
    axes[0, 0].imshow(img_resized)
    axes[0, 0].set_title(f"Original  |  GT: {gt_label}", fontsize=10)

    im1 = axes[0, 1].imshow(norm_map, cmap="magma")
    axes[0, 1].set_title("Backbone: Patch Token L2 Norm", fontsize=10)
    fig.colorbar(im1, ax=axes[0, 1], fraction=0.046, pad=0.04)

    axes[0, 2].imshow(pca_rgb)
    axes[0, 2].set_title("Backbone: PCA RGB Map", fontsize=10)

    # Row 1 — attention head view
    axes[1, 0].imshow(img_resized)
    axes[1, 0].set_title(f"Pred: {pred_label} ({prob:.3f}) {correct}", fontsize=10)

    im2 = axes[1, 1].imshow(attn_2d, cmap="jet", vmin=0, vmax=1)
    axes[1, 1].set_title("Head: Gated Attention Map", fontsize=10)
    fig.colorbar(im2, ax=axes[1, 1], fraction=0.046, pad=0.04)

    axes[1, 2].imshow(img_resized)
    axes[1, 2].imshow(attn_img, cmap="jet", alpha=0.5)
    axes[1, 2].set_title("Head: Attention Overlay", fontsize=10)

    for ax in axes.flatten(): ax.axis("off")

    plt.suptitle(
        f"Ablation A — ImageNet | {POOLING_MODE.upper()} | Sample {idx}",
        fontsize=11, fontweight="bold"
    )
    plt.tight_layout()
    plt.savefig(f"{out_dir}/dashboard_{idx}_gt{int(label)}.png", bbox_inches="tight")
    plt.close(fig)

# =========================
# MAIN EVALUATION
# =========================
def evaluate():
    test_loader = DataLoader(
        FociEvalDataset(CFG["test_csv"]),
        batch_size=CFG["batch_size"],
        shuffle=False,
        collate_fn=safe_collate_eval,
        num_workers=0
    )

    model = FociViT_Eval(CFG["backbone_name"], CFG["image_size"], POOLING_MODE).to(DEVICE)

    model.load_state_dict(
        torch.load(CFG["model_ckpt"], map_location=DEVICE),
        strict=False
    )
    model.eval()
    print("Model loaded successfully")

    all_probs, all_labels = [], []
    vis_count = 0

    print("Running inference...")
    with torch.no_grad():
        for batch in tqdm(test_loader):
            if batch is None: continue
            x, y, paths = batch
            x, y = x.to(DEVICE), y.to(DEVICE)

            probs, spatial_maps, patch_tokens_batch = model(x)

            all_probs.extend(probs.cpu().numpy())
            all_labels.extend(y.cpu().numpy())

            if SAVE_VISUALIZATIONS and spatial_maps is not None \
                    and vis_count < NUM_VIS_TO_SAVE:
                for i in range(x.size(0)):
                    if vis_count >= NUM_VIS_TO_SAVE: break
                    save_dashboard(
                        paths[i], patch_tokens_batch[i],
                        spatial_maps[i], probs[i].item(),
                        y[i].item(), vis_count, CFG["output_dir"]
                    )
                    vis_count += 1

    y_true  = np.array(all_labels).astype(int)
    y_probs = np.array(all_probs)
    y_preds = (y_probs >= 0.5).astype(int)

    acc     = accuracy_score(y_true, y_preds)
    prec    = precision_score(y_true, y_preds, zero_division=0)
    rec     = recall_score(y_true, y_preds, zero_division=0)
    f1      = f1_score(y_true, y_preds, zero_division=0)
    roc_auc = auc(*roc_curve(y_true, y_probs)[:2])

    print("\n" + "="*42)
    print(f"  RESULTS — {POOLING_MODE.upper()} V02")
    print("="*42)
    print(f"  Accuracy  : {acc:.4f}")
    print(f"  Precision : {prec:.4f}")
    print(f"  Recall    : {rec:.4f}")
    print(f"  F1 Score  : {f1:.4f}")
    print(f"  ROC AUC   : {roc_auc:.4f}")
    print("="*42)

    with open(f"{CFG['output_dir']}/metrics.txt", "w") as f:
        f.write(f"Model      : {POOLING_MODE} V02 (Two-Stage + LayerNorm)\n")
        f.write(f"Checkpoint : {CFG['model_ckpt']}\n")
        f.write(f"Test CSV   : {CFG['test_csv']}\n\n")
        f.write(f"Accuracy   : {acc:.4f}\n")
        f.write(f"Precision  : {prec:.4f}\n")
        f.write(f"Recall     : {rec:.4f}\n")
        f.write(f"F1 Score   : {f1:.4f}\n")
        f.write(f"ROC AUC    : {roc_auc:.4f}\n")

    # ── Confusion matrix ─────────────────────────────────────────────────────
    cm = confusion_matrix(y_true, y_preds)
    plt.figure(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=["Healthy", "Damaged"],
                yticklabels=["Healthy", "Damaged"])
    plt.xlabel("Predicted"); plt.ylabel("Ground Truth")
    plt.title(f"Confusion Matrix — {POOLING_MODE.upper()} V02")
    plt.tight_layout()
    plt.savefig(f"{CFG['output_dir']}/confusion_matrix.png")
    plt.close()

    # ── ROC curve ────────────────────────────────────────────────────────────
    fpr, tpr, _ = roc_curve(y_true, y_probs)
    plt.figure(figsize=(6, 6))
    plt.plot(fpr, tpr, color="darkorange", lw=2, label=f"AUC = {roc_auc:.3f}")
    plt.plot([0, 1], [0, 1], color="navy", lw=2, linestyle="--")
    plt.xlabel("False Positive Rate"); plt.ylabel("True Positive Rate")
    plt.title(f"ROC Curve — {POOLING_MODE.upper()} V02")
    plt.legend(loc="lower right")
    plt.tight_layout()
    plt.savefig(f"{CFG['output_dir']}/roc_curve.png")
    plt.close()

    print(f"\nDone. All results saved to:\n{CFG['output_dir']}")


if __name__ == "__main__":
    evaluate()